# Printed-Range Trap: model runs

Runs every model and harness listed in `RUNS`, one after the other. All the logic is in `src/` of the repository.

Kaggle settings: GPU T4 x2, internet on, `HF_TOKEN` secret attached (needed for MedGemma).

For long runs use **Save Version > Save & Run All (Commit)**: it keeps running if the browser is closed and keeps the output files.
Runs are resumable: finished runs are skipped and an interrupted run continues from its last saved prompt.

## Runs

In [ ]:
RUNS = [
    ("qwen", "instruction"),
    ("qwen", "guidelines"),
    ("qwen", "agent"),
    ("medgemma", "instruction"),
    ("medgemma", "guidelines"),
]

## Setup

In [ ]:
!pip install -q uv
!uv pip install --system -q "transformers==5.18.0" "accelerate==1.15.0"

In [ ]:
import os
import shutil

import torch
import transformers
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), "No GPU: set the accelerator to GPU T4 x2"
assert transformers.__version__ == "5.18.0", "Old transformers still loaded: restart the session"

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:  # noqa: BLE001 - no secret attached; fine for Qwen
    print("No HF_TOKEN secret (needed for MedGemma)")

REPO_DIR = "/kaggle/working/printed-range-trap"
shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone -q https://github.com/REZZOUGAICHA/printed-range-trap.git {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --format="code version: %h %s"

## Prompts

In [ ]:
for harness in ["plain", "instruction", "guidelines"]:
    !python src/build_prompts.py --languages en fr --harness {harness}
!git diff --quiet data/ && echo "prompts match the repository" || echo "WARNING: prompts differ from the repository"

## Models

In [ ]:
MODELS = {
    "qwen": (
        "qwen3.5-4b",
        (
            "--model-id Qwen/Qwen3.5-4B --revision 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a "
            "--dtype float16 --device-map cuda:0"
        ),
    ),
    # MedGemma: float16 overflows on T4 (NaN logits), and its thinking token is suppressed
    "medgemma": (
        "medgemma-1.5-4b",
        (
            "--model-id google/medgemma-1.5-4b-it --revision 91850547d9f0b2fdd21aa7c5f4f3d1a8a52c243b "
            "--dtype float32 --device-map auto --suppress-tokens '<unused94>'"
        ),
    ),
}

# Saved outside the code folder, so re-running the setup cell never deletes answers
RESULTS = "/kaggle/working/results"


def run_config(model, harness):
    name, args = MODELS[model]
    assert harness != "agent" or model == "qwen", "the agent harness needs tool support: Qwen only"
    script = "src/run_agent.py" if harness == "agent" else "src/run_model.py"
    prompts = (
        "data/prompts.jsonl" if harness in ("plain", "agent") else f"data/prompts_{harness}.jsonl"
    )
    if harness == "plain":
        out = f"{RESULTS}/{name}/generations_{name}.jsonl"
    else:
        out = f"{RESULTS}/{name}/{harness}/generations_{name}_{harness}.jsonl"
    return f"{script} {args} --prompts {prompts} --out {out}", out

## Run

In [ ]:
outputs = []
for model, harness in RUNS:
    command, out = run_config(model, harness)
    print(f"\n===== {model} / {harness} =====")
    !python {command}
    outputs.append(out)

## Results

In [ ]:
for out in outputs:
    if os.path.exists(out):
        print(f"\n===== {out.split('/')[-1]}")
        !python src/metrics.py {out} | grep -E "deference_range_printed|format_failure"
    else:
        print(f"\nmissing: {out}")

## Download
Unzip into the root of the local repository: the files land in `results/`.

In [ ]:
from IPython.display import FileLink

os.chdir("/kaggle/working")
!rm -f results.zip && zip -r -q results.zip results
FileLink("results.zip")